In [ ]:
import os
import glob
import numpy as np
import xarray as xr

import matplotlib
from matplotlib import cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline

from joblib import Parallel, delayed

# --- gridded NetCDF + per-basin river profiles (PyGMT/ArcGIS + matplotlib) ---
from gospl.analyse.gridexport import (
    grid_export, to_netcdf, basin_rivers, plot_long_profile, plot_basin_map)

# --- stratigraphic sections / wells / Wheeler (matplotlib, inline) ---
from gospl.analyse.stratasection import (
    load_strata, cross_section, horizontal_slice, synthetic_well, wheeler,
    well_panel)

## Running the simulation

First activate the conda environment:

```bash
conda activate gospl
```

To run the simulation in a terminal (`X` = number of MPI processes, e.g. 5):

```bash
mpirun -np X gospl -i input-escarpment.yml
```

# Analysing the outputs

All the post-processing below uses goSPL's built-in **`gospl.analyse`** toolkit
(imported in the first cell). Two complementary modules are used:

- **`gospl.analyse.gridexport`** — reassembles the unstructured mesh, rasterises
  every surface field of an output step onto a **regular grid**, runs D8
  hydrology (drainage area, basins, &chi;) and writes a CF-NetCDF for
  PyGMT/ArcGIS. It also exposes per-basin **river long-profile** helpers.
- **`gospl.analyse.stratasection`** — reads the recorded stratigraphy and draws
  **cross-sections, synthetic wells and Wheeler (chronostratigraphic)
  diagrams** (coloured by facies, lithology, provenance, &hellip;).

Every function used below has a **terminal equivalent** so the same products can
be generated outside Jupyter. These console commands &mdash; `gospl-grid`,
`gospl-section`, `gospl-strata-volume` &mdash; are installed with goSPL and are
shown in each section.

Each gridded NetCDF (one file per output step) holds, when available (every
variable carries its `units` and a `long_name` definition):

+ surface elevation `elev` (m) and the step's `sea_level` (m)
+ cumulative erosion/deposition `erodep` (m) and its rate `EDrate` (m/yr)
+ water / sediment fluxes `FA`, `fillFA`, `waterFill`, `sedLoad`
+ hydrology: `drainage_area`, `basin` id, `chi`, `flowdist`, and the
  priority-flood-`filled` elevation

In [ ]:
# Define output folder name for the simulation
out_path = 'results/'

if not os.path.exists(out_path):
    os.makedirs(out_path)

### Rasterising the outputs to a regular grid &mdash; `grid_export` / `to_netcdf`

`grid_export` reassembles the global mesh, interpolates a step's fields onto a
regular grid, runs the D8 hydrology and returns a dict of 2-D arrays;
`to_netcdf` writes that to a CF-NetCDF (each variable annotated with its `units`
and `long_name`). `getOutputs` below simply loops over the steps and writes one
`results/surface<step>.nc` per step.

**`grid_export(h5dir, mesh, step=None, ...)` &mdash; main options**

| Argument | Default | Meaning |
|---|---|---|
| `h5dir` | &ndash; | the run's `h5` output directory |
| `mesh` | &ndash; | global mesh `.npz` (vertices `v`, cells `c`) |
| `step` | last | output step to rasterise |
| `spacing` | median edge | grid resolution `dx[,dy]` (mesh units) |
| `fields` | all | subset of surface fields to include |
| `mn` | `0.5` | &chi; concavity `m/n` |
| `a0` | `1.0` | &chi; reference drainage area |
| `base_level` | run sea level | elevation defining the coast / outlets (catchment + &chi; datum) |
| `latlim` | `89` | (global meshes) crop the polar caps |

Global (spherical) meshes are auto-detected and gridded in lon/lat. The resolved
sea level is stored in each file (global attribute **and** a `sea_level`
variable), so the grid is self-describing.

**Terminal equivalent** (one step &rarr; one NetCDF):

```bash
gospl-grid --h5dir escarpment/h5 --mesh data/escarpment.npz:v:c \
    --step 50 --spacing 250 --out results/surface25.nc
```

For the whole time series, loop in the shell:

```bash
for s in $(seq 0 50); do
  gospl-grid --h5dir escarpment/h5 --mesh data/escarpment.npz:v:c \
      --step $s --spacing 250 --out results/surface$s.nc
done
```

In [ ]:
h5dir = "escarpment/h5"
mesh = "data/escarpment.npz"
reso = 500

out_name = "surface"

def getOutputs(steps):

    # clear any stale .nc files first
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return
        
    for stp in steps:
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)                       

    return

def getOutputsParallel(steps, h5dir, n_workers=8):
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return

    def process_step(stp):
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)

    Parallel(n_jobs=n_workers)(delayed(process_step)(stp) for stp in steps)

steps = np.arange(51)
getOutputsParallel(steps, h5dir, n_workers=8)
# getOutputs(steps)

In [ ]:
# h5dir = "escarpment-oro/h5"
# out_name = "oro"
# getOutputsParallel(steps, h5dir, n_workers=8)

### Surface elevation through time

The four panels show the remapped `elevation` field at steps 5, 10, 15 and 25, with the black contour marking the $0$ m shoreline. Watch how the coastline migrates as the prescribed sea level and sediment supply reshape the margin: a seaward-stepping shoreline indicates progradation, a landward-stepping one indicates transgression.

In [ ]:
out_name = "surface"
ncfiles = [os.path.join(out_path, f"{out_name}{stp}.nc") for stp in steps]
ds = {stp: xr.open_dataset(f) for stp, f in zip(steps, ncfiles)}
ds[5]

In [ ]:
# out_name = "oro"
# ncfiles = [os.path.join(out_path, f"{out_name}{stp}.nc") for stp in steps]
# ds_oro = {stp: xr.open_dataset(f) for stp, f in zip(steps, ncfiles)}
# ds_oro[5]

In [ ]:
stps = [10,25,35,50]
fig, axs = plt.subplots(2,2, figsize=(8,8), sharex=True, sharey=True)
for ax, stp in zip(axs.flat, stps):
    im = ds[stp].elev.plot(ax=ax, add_labels=False, add_colorbar=False, cmap='Spectral_r')
    ds[stp].elev.plot.contour(ax=ax, levels=[ds[stp].sea_level.values], colors=['k'])
for ax, stp in zip(axs.flat, stps):
    ax.set_title(f'step = {stp}', fontsize=10, fontweight="bold")
cbar_ax = fig.add_axes([0.2, -0.02, 0.6, 0.02]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Elevation (m)')
plt.show()

fig, axs = plt.subplots(2,2, figsize=(8,8), sharex=True, sharey=True)
for ax, stp in zip(axs.flat, stps):
    im = ds[stp].EDrate.plot(ax=ax, add_labels=False, add_colorbar=False, cmap='bwr')
    ds[stp].elev.plot.contour(ax=ax, levels=[ds[stp].sea_level.values], colors=['k'])
for ax, stp in zip(axs.flat, stps):
    ax.set_title(f'step = {stp}', fontsize=10, fontweight="bold")
cbar_ax = fig.add_axes([0.2, -0.02, 0.6, 0.02]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Erosion / deposition rates (m/yr)')
plt.show()

### Longitudinal profile evolution

We collapse each step's grid to a mean profile along $y$ and overlay every second step in grey, with step 0 (red) and step 25 (blue) emphasised. The shaded region between the initial and final profiles shows the net change: where the final profile sits below the initial one the margin has aggraded/prograded a sediment wedge, illustrating how the depositional surface builds basinward over the $250$ kyr run.

In [ ]:
stp = 50
meands = ds[stp].mean(dim='x')
maxds = ds[stp].max(dim='x')
minds = ds[stp].min(dim='x')

plt.figure(figsize=(8,4))
ax = plt.gca()

meands.elev.plot(lw=2,c='k',label='mean')
maxds.elev.plot(lw=1,c='b',ls='-.',label='max')
minds.elev.plot(lw=1,c='r',ls='-.',label='min')

plt.xlabel('Distance y-axis (m)')
plt.ylabel('elevation (m)')
plt.title('Averaged longitudinal profile at the last time step', size=10)
plt.xlim(ds[stp].x.min(),ds[stp].x.max())
plt.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
means = {stp: d.mean(dim='x') for stp, d in ds.items()}

fig, ax = plt.subplots(figsize=(8, 4))
cmap = plt.get_cmap('gray_r', len(ds) + 1)

for k in range(0, len(ds), 2):
    means[k].elev.plot(ax=ax, lw=1, ls='-.', c=cmap(k))

means[0].elev.plot(ax=ax, lw=2, c='r', label='step 0')
means[stp].elev.plot(ax=ax, lw=2, c='b', label='step 25')

minz = np.minimum(means[stp].elev, means[0].elev)
ax.fill_between(means[0].y, -520, minz, facecolor='gainsboro')

ax.set(
    xlabel='Distance y-axis (m)', ylabel='Elevation (m)',
    title='Averaged longitudinal profile through time',
    xlim=(means[0].y.min(), means[0].y.max()),
    ylim=(minz.min()-10, 1300),
)
ax.legend(frameon=False, loc='upper right')
plt.tight_layout()
plt.show()

## Drainage basins and river long profiles

The gridded files already carry the `basin` ids and `chi`. To examine the
**channel network of a single basin**, `basin_rivers` traces the main stem (up
the largest-area donor at each step) and its tributaries; `plot_basin_map` maps
them with the **sea-level coastline**, and `plot_long_profile` draws the
longitudinal profile (distance in km).

**What you can do:** pick a basin (by id, or by clicking a point as below), set
the channel-defining drainage-area threshold, map the network over any gridded
field, and plot the long profile against distance or **&chi;** &mdash; either the
raw elevation (keeps real lakes as dips) or the hydrologically-`filled` one
(monotonic).

| Function | Key options | Meaning |
|---|---|---|
| `basin_rivers(result, ...)` | `basin_id`, `area_threshold` | basin to extract (default: largest); min drainage area (m&sup2;) for a channel (default: 95th pct) |
| `plot_basin_map(result, rivers, ...)` | `background`, `sea_level`, `figsize` | base field (default `elev`); coastline datum (default run sea level); figure size |
| `plot_long_profile(rivers, ...)` | `xaxis`, `which`, `figsize` | `dist` or `chi`; `elev` (raw) or `filled` (monotonic); figure size |

These per-basin plots are a **notebook API** (no dedicated console command); they
read the same grid `gospl-grid` produces. Below we first locate a basin id by
picking a point, then extract and plot its rivers.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

elev = ds[stp].elev
elev.plot(ax=ax, cmap="gray", alpha=0.2, add_colorbar=False)

basin = ds[stp].basin
basin.plot(ax=ax, cmap="jet") #,vmax=300)
levels = np.unique(ds[stp].basin.values)
cs = ax.contour(basin.x, basin.y, basin.values, levels=levels, colors="k", linewidths=0.1)
cs = ax.contour(basin.x, basin.y, elev.values, levels=[ds[stp].sea_level], colors="k", linestyles='-', linewidths=1)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

elev = ds[stp].elev
elev.plot(ax=ax, cmap="gray", alpha=0.2, add_colorbar=False)

chi = ds[stp].chi
chi.plot(ax=ax, cmap="spring", vmax=10)
levels = np.unique(ds[stp].basin.values)
cs = ax.contour(basin.x, basin.y, basin.values, levels=levels, colors="k", linewidths=0.1)
cs = ax.contour(basin.x, basin.y, elev.values, levels=[ds[stp].sea_level], colors="k", linestyles='-', linewidths=1)

plt.tight_layout()
plt.show()

In [ ]:
xbasin = 190.e3
ybasin = 150.e3
basin_id = int(ds[stp].sel(x=xbasin, y=ybasin, method='nearest').basin.values)
print(f"Corresponding basin ID: {basin_id}")

In [ ]:
g = grid_export(h5dir, mesh, stp, spacing=reso)
riv = basin_rivers(g, basin_id=basin_id, area_threshold=5e6)
plot_basin_map(g, riv)
plot_long_profile(riv, which="elev")

## Scarp retreat through time

Faint grey lines are the cross-escarpment profiles at every Myr; bold black lines mark every 5th step plus the final profile. The thick coloured tracks join the crest positions over time (mean/min/max), so their inland slope is the **retreat rate** of the scarp crest. A steepening or flattening of these tracks reveals acceleration or stalling of the retreat.

The same crest-position tracks, now shown against just three representative profiles (steps 1, 30, 49) to keep the figure readable while still conveying the overall retreat trajectory.

In [ ]:
xbasin = 190.e3
ybasin = 150.e3
stp2 = 40
basin_id2 = int(ds[stp2].sel(x=xbasin, y=ybasin, method='nearest').basin.values)
print(f"Corresponding basin ID: {basin_id2}")
g2 = grid_export(h5dir, mesh, stp2, spacing=reso)
riv2 = basin_rivers(g2, basin_id=basin_id2, area_threshold=5e6)
plot_basin_map(g2, riv2)
plot_long_profile(riv2, which="elev")
plot_long_profile(riv, which="elev")

## Escarpment retreat

As erosion strips mass from the margin, the lithosphere rebounds isostatically. goSPL solves this with a finite-difference flexure model (elastic thickness $T_e = 20$ km, crust/mantle densities 2300/3300 kg m$^{-3}$, $E = 65$ GPa, $\nu = 0.25$). The exported `flex` field is the resulting vertical deflection. Here we reload each step's cross-escarpment (mean/max/min) profiles to examine the deflection signal.

In [ ]:
meands = ds[stp].mean(dim='x')
maxds = ds[stp].max(dim='x')
minds = ds[stp].min(dim='x')

escarpment_pos_mean = meands.elev.where(meands.elev==meands.elev.max(), drop=True).squeeze().y.values
escarpment_pos_max = maxds.elev.where(maxds.elev==maxds.elev.max(), drop=True).squeeze().y.values
escarpment_pos_min = minds.elev.where(minds.elev==minds.elev.max(), drop=True).squeeze().y.values


**What to look for:** each curve is the *change* in flexural deflection between consecutive steps (m/Myr), i.e. the instantaneous isostatic uplift rate along the profile. Positive values where the scarp is eroding indicate rock uplift in response to unloading; the pattern migrates inland following the scarp crest, reinforcing the retreat by keeping the margin elevated.

In [ ]:

plt.figure(figsize=(8,4))
ax = plt.gca()

meands.elev.plot(lw=2,c='k',label='mean')
maxds.elev.plot(lw=1,c='b',ls='-.',label='max')
minds.elev.plot(lw=1,c='r',ls='-.',label='min')

plt.scatter(escarpment_pos_mean,meands.elev.max(),c='w',s=30,edgecolors='k',zorder=2)
plt.scatter(escarpment_pos_min,minds.elev.max(),c='w',s=30,edgecolors='r',zorder=2)
plt.scatter(escarpment_pos_max,maxds.elev.max(),c='w',s=30,edgecolors='b',zorder=2)

plt.xlabel('Distance y-axis (m)')
plt.ylabel('elevation (m)')
plt.title('Cross-escarpment profile for a specific time step', size=10)
plt.ylim(0,1800)
plt.xlim(ds[stp].y.min(),ds[stp].y.max())
plt.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
escpt_mean = []
escpt_max = []
escpt_min = []

escpt_pos_mean = []
escpt_pos_max = []
escpt_pos_min = []

for s in range(1,51):
    # Get escarpment longitudinal profile information
    meands = ds[s].mean(dim='x')
    maxds = ds[s].max(dim='x')
    minds = ds[s].min(dim='x')
    escpt_mean.append(meands)
    escpt_max.append(maxds)
    escpt_min.append(minds)

    # Get escarpment position
    maxz_mean = meands.elev.max().values
    maxz_min = minds.elev.max().values
    maxz_max = maxds.elev.max().values

    escarpment_pos_mean = meands.elev.where(meands.elev==maxz_mean, drop=True).squeeze().y.values
    escarpment_pos_max = maxds.elev.where(maxds.elev==maxz_max, drop=True).squeeze().y.values
    escarpment_pos_min = minds.elev.where(minds.elev==maxz_min, drop=True).squeeze().y.values
    
    escpt_pos_mean.append([escarpment_pos_mean,maxz_mean])
    escpt_pos_max.append([escarpment_pos_max,maxz_max])
    escpt_pos_min.append([escarpment_pos_min,maxz_min])

Generate a plot to visualize results.

In [ ]:
plt.figure(figsize=(8,3))
ax = plt.gca()

cmap = plt.get_cmap('gray_r', len(escpt_mean)+1) 
    
for k in range(0,len(escpt_mean)):
    meands = escpt_mean[k]
    escarpment_pos_mean = escpt_pos_mean[k]
    meands.elev.plot(lw=1,ls='-.',c=cmap(k),alpha=0.2)
    plt.scatter(escarpment_pos_mean[0],escarpment_pos_mean[1],c='w',
                s=30,edgecolors=cmap(k),zorder=2,alpha=0.2)


for k in range(0,len(escpt_mean),5):
    meands = escpt_mean[k]
    escarpment_pos_mean = escpt_pos_mean[k]
    meands.elev.plot(lw=1,c='k')
    plt.scatter(escarpment_pos_mean[0],escarpment_pos_mean[1],c='w',
                s=30,edgecolors=cmap(k),zorder=4)

k = 49
meands = escpt_mean[k]
escarpment_pos_mean = escpt_pos_mean[k]
meands.elev.plot(lw=0.5,c='k')
plt.scatter(escarpment_pos_mean[0],escarpment_pos_mean[1],c='w',
            s=30,edgecolors=cmap(k),zorder=4)

val = np.asarray(escpt_pos_mean)
plt.plot(val[:,0],val[:,1],lw=3,c='k',label='mean')
val = np.asarray(escpt_pos_min)
plt.plot(val[:,0],val[:,1],lw=2,c='r',label='min')
val = np.asarray(escpt_pos_max)
plt.plot(val[1:,0],val[1:,1],lw=2,c='b',label='max')

plt.xlabel('Distance y-axis (m)')
plt.ylabel('elevation (m)')
plt.title('Cross-escarpment profile through time', size=10)
plt.ylim(0,1800)
# plt.xlim(dataset.y.min(),dataset.y.max())
plt.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## Flexural response

In [ ]:
flex_mean = []
flex_max = []
flex_min = []

for s in range(1,50):
    
    # Get escarpment longitudinal profile information
    meands = ds[s].mean(dim='x')
    maxds = ds[s].max(dim='x')
    minds = ds[s].min(dim='x')
    flex_mean.append(meands)
    flex_max.append(maxds)
    flex_min.append(minds)

In [ ]:
plt.figure(figsize=(8,3))
ax = plt.gca()

cmap = plt.get_cmap('Blues', 51) 

for k in range(1,49):
    val = flex_mean[k].flexIso-flex_mean[k-1].flexIso
    val.plot(lw=0.4,ls='-.',c='k',alpha=0.3)
    if k%5 == 0:
        val.plot(lw=2,c=cmap(k),label='step '+str(k))

plt.xlabel('Distance y-axis (m)')
plt.ylabel('Flexural response (m/Myr)')
plt.title('Cross-escarpment isostatic adjustments through time', size=10)
plt.xlim(ds[0].y.min()+750,ds[0].y.max()-750)
plt.legend(frameon=False, loc="upper right",bbox_to_anchor=(1.15, 1.05), fontsize=8)
plt.tight_layout()
plt.show()